# Regenerate Figure A5 (the repetition counterexample structures)

**The bug.** `fig1_structures.py` colours by the PDB B-factor column with
`spectrum("b", "red_white_blue", minimum=20, maximum=90)`, which assumes pLDDT on a
0-100 scale. The B-factors in these PDB files are on a **0-1 scale**, so every atom
falls below the `minimum=20` cutoff and PyMOL clamps the whole structure to red.
That is why both panels render solid red even though the left chain has mean pLDDT 85.87.

**The fix.** Colour from the per-residue pLDDT CSVs instead. Those hold the authoritative
0-100 values that match the paper (means 85.875 and 28.846, and the min/mean/max in
Table A16), and they avoid the B-factor column's rounding.

**Upload these 4 files** as a Kaggle Dataset (or into `/kaggle/input/`), all of which sit
together in `ai4dd-plm-steering/analysis/fig1/`:

- `fig1_ubiquitin_repeat.pdb`
- `fig1_eerhv_repeat.pdb`
- `fig1_ubiquitin_repeat_per_residue_plddt.csv`
- `fig1_eerhv_repeat_per_residue_plddt.csv`

Internet must be ON (Settings -> Internet) for the install cell. No GPU needed.

## 1. Install PyMOL

In [ ]:
!pip install -q pymol-open-source 2>&1 | tail -2
import pymol
print("pymol", pymol.__file__)

## 2. Find the input files

Searches `/kaggle/input` recursively, so the dataset folder name does not matter.

In [ ]:
import os, shutil, glob

NEEDED = ["fig1_ubiquitin_repeat.pdb",
          "fig1_eerhv_repeat.pdb",
          "fig1_ubiquitin_repeat_per_residue_plddt.csv",
          "fig1_eerhv_repeat_per_residue_plddt.csv"]

WORK = "/kaggle/working"
found = {}
for name in NEEDED:
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if hits:
        found[name] = hits[0]
        shutil.copy(hits[0], os.path.join(WORK, name))

missing = [n for n in NEEDED if n not in found]
if missing:
    raise SystemExit("MISSING: " + ", ".join(missing) +
                     "\nUpload them to /kaggle/input and re-run this cell.")

for n, p in found.items():
    print(f"ok  {n:50s} <- {p}")

## 3. Check the scales

Confirms the diagnosis before rendering. The CSV values should span roughly 12-96
(0-100 scale) and the PDB B-factors roughly 0.08-0.99 (0-1 scale). The CSV means must
read 85.87 and 28.85, matching the paper.

In [ ]:
import csv

def read_plddt(path):
    rows = list(csv.DictReader(open(path)))
    key = [c for c in rows[0] if "plddt" in c.lower()][0]
    return {int(r["residue_index"]): float(r[key]) for r in rows}

plddt = {
    "confident": read_plddt("fig1_ubiquitin_repeat_per_residue_plddt.csv"),
    "collapsed": read_plddt("fig1_eerhv_repeat_per_residue_plddt.csv"),
}

for name, d in plddt.items():
    v = list(d.values())
    print(f"{name:10s} CSV per-residue pLDDT: n={len(v)} "
          f"min={min(v):.2f} mean={sum(v)/len(v):.2f} max={max(v):.2f}")

for name, pdb in (("confident", "fig1_ubiquitin_repeat.pdb"),
                  ("collapsed", "fig1_eerhv_repeat.pdb")):
    b = [float(l[60:66]) for l in open(pdb) if l.startswith("ATOM")]
    print(f"{name:10s} PDB B-factor column : n={len(b)} "
          f"min={min(b):.3f} mean={sum(b)/len(b):.3f} max={max(b):.3f}   <- 0-1 scale, the bug")

assert abs(sum(plddt["confident"].values()) / 258 - 85.875) < 0.01, "confident mean should be 85.87"
assert abs(sum(plddt["collapsed"].values()) / 135 - 28.846) < 0.01, "collapsed mean should be 28.85"
print("\nCSV means match the paper (85.87 / 28.85).")

## 4. Render both panels

Everything below is the original `fig1_structures.py`, unchanged except that per-residue
pLDDT is written into the B-factor column from the CSV before the spectrum call.

The shared-scale guarantee is preserved: both molecules are centred on the origin, one
camera view is locked with `get_view`/`set_view`, and both panels render at 900x900. So
Angstroms-per-pixel is identical in the two panels, and the collapsed 135-mer (Rg 24.0 A)
must look *larger* than the confident 258-mer (Rg 22.2 A). That size contrast is the figure.

In [ ]:
import pymol
pymol.finish_launching(["pymol", "-cq"])
from pymol import cmd

cmd.reinitialize()
cmd.bg_color("white")
cmd.set("ray_opaque_background", 1)
cmd.set("ray_shadows", 0)
cmd.set("antialias", 2)
cmd.set("cartoon_transparency", 0)

cmd.load("fig1_ubiquitin_repeat.pdb", "confident")   # 258 res, pLDDT 85.87 (compact)
cmd.load("fig1_eerhv_repeat.pdb",     "collapsed")   # 135 res, pLDDT 28.85 (extended)
cmd.hide("everything")
cmd.show("cartoon")

# ---- THE FIX ---------------------------------------------------------------
# Overwrite the B-factor column with per-residue pLDDT on the 0-100 scale, so the
# fixed spectrum range below means what it says.
for obj, d in plddt.items():
    for resi, val in d.items():
        cmd.alter(f"{obj} and resi {resi}", f"b={val}")
cmd.sort()
cmd.rebuild()

for obj in ("confident", "collapsed"):
    bs = [a.b for a in cmd.get_model(obj).atom]
    print(f"{obj:10s} B after fix: min={min(bs):.2f} mean={sum(bs)/len(bs):.2f} max={max(bs):.2f}")
# -----------------------------------------------------------------------------

# blue = confident (high pLDDT), red = low confidence. SAME fixed range on both.
for obj in ("confident", "collapsed"):
    cmd.spectrum("b", "red_white_blue", obj, minimum=20, maximum=90)

# centre BOTH molecules on the origin so one shared camera frames each centrally
for obj in ("confident", "collapsed"):
    c = cmd.centerofmass(obj)
    cmd.translate([-c[0], -c[1], -c[2]], obj, camera=0)

# fix the shared SCALE on the LARGER (collapsed) molecule, then lock the view
cmd.orient("collapsed")
cmd.zoom("collapsed", 10)          # 10 A padding
view = cmd.get_view()              # the camera distance stored here == the shared scale

W = H = 900
cmd.disable("collapsed"); cmd.enable("confident")
cmd.set_view(view); cmd.ray(W, H); cmd.png("panel_confident.png", dpi=300)

cmd.disable("confident"); cmd.enable("collapsed")
cmd.set_view(view); cmd.ray(W, H); cmd.png("panel_collapsed.png", dpi=300)

print("wrote panel_confident.png and panel_collapsed.png")

## 5. Stitch into one PDF

The assert is the shared-scale tripwire from `stitch_fig1_structures.py`: identical view
and identical pixel size means the panels must be the same size here. If they are not,
the scale drifted and the output must not be used.

In [ ]:
from PIL import Image

a = Image.open("panel_confident.png").convert("RGB")   # left  = confident 258-mer
b = Image.open("panel_collapsed.png").convert("RGB")   # right = collapsed 135-mer
assert a.size == b.size, "panels differ in size -> scale is NOT shared, stop"

canvas = Image.new("RGB", (a.width + b.width, a.height), "white")
canvas.paste(a, (0, 0))
canvas.paste(b, (a.width, 0))
canvas.save("fig1_structures.pdf", "PDF", resolution=300.0)
print("wrote fig1_structures.pdf", canvas.size)

## 6. Check it before you use it

**The left panel must be mostly blue and the right panel mostly red.** If both are still
red, the fix did not apply - do not upload the output.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.imshow(canvas)
ax.axis("off")
ax.set_title("left: confident, pLDDT 85.87 (expect BLUE)      "
             "right: collapsed, pLDDT 28.85 (expect RED)", fontsize=10)
plt.show()

## 7. Download

`fig1_structures.pdf` appears in the notebook's Output panel on the right. Download it, then:

1. Upload it to Prism, replacing the existing `figA1_structures.pdf` (the paper's Figure A5).
2. Save it locally over
   `final/plm-steering-audit/analysis/final/figA1_structures.pdf`, then commit and push.

The caption needs no change. Once the colours are right, "red = low confidence,
blue = high" is finally true of the image.

In [ ]:
from IPython.display import FileLink
import os

print(os.path.abspath("fig1_structures.pdf"),
      os.path.getsize("fig1_structures.pdf"), "bytes")
FileLink("fig1_structures.pdf")